In [15]:
!pip install numpy
import numpy as np

In [1]:
import chipwhisperer as cw
import os, time

# (Assuming scope and target are already connected)
#bitstream_path =  r'C:\Users\sbista\ChipWhisperer\chipwhisperer\firmware\fpgas\aes\vivado_\cw305_aes.runs\impl_100t\cw305_top.bit'
bitstream_path =  r'/home/sareeta/chipwhisperer/firmware/fpgas/aes/vivado_is/vivado/cw305_aes.runs/impl_100t/cw305_top.bit'
assert os.path.isfile(bitstream_path), f"Bitstream not found: {bitstream_path}"

# 2) Connect to the capture board (CWLite)
scope = cw.scope()
scope.default_setup()

# 3) Connect and Program the FPGA
print("Programming CW305 FPGA with:", bitstream_path)
target = cw.target(scope, cw.targets.CW305, bsfile=bitstream_path, force=True)


(ChipWhisperer NAEUSB WARNING|File naeusb.py:826) Your firmware (0.51.0) is outdated - latest is 0.54.0 See https://chipwhisperer.readthedocs.io/en/latest/firmware.html for more information


scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 1310998                   to 12047007                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 4169666                   to 29538459                 
scope.clock.adc_rate                     changed from 4169666.0                 to 29538459.0               
scope.clock.freq_ct

(ChipWhisperer Target WARNING|File CW305.py:591) Using default Verilog defines (/home/sareeta/chipwhisperer/software/chipwhisperer/hardware/firmware/cw305/cw305_aes_defines.v); if this is not what you want, provide them via the defines_files argument


In [6]:
from Crypto.Cipher import AES

# Known NIST AES-128 test vector
KEY = bytes.fromhex("2b7e151628aed2a6abf7158809cf4f3c")
PT  = bytes.fromhex("5c692f9103b2302914d7e555e4dcee69")

# Expected ciphertext
EXP_CT = AES.new(KEY, AES.MODE_ECB).encrypt(PT)
print("Expected CT:", EXP_CT.hex())


Expected CT: b584c14c1c21746f1234457996b62b7c


In [7]:
# Write KEY
target.fpga_write(target.REG_CRYPT_KEY, KEY)

# Write PLAINTEXT
target.fpga_write(target.REG_CRYPT_TEXTIN, PT)

# Trigger encryption
target.fpga_write(target.REG_CRYPT_GO, b"\x01")

# Small delay (AES is fast but be safe)
time.sleep(0.01)

In [8]:
# Read ciphertext
ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))
print("FPGA CT   :", ct.hex())
print("MATCH?    :", ct == EXP_CT)

FPGA CT   : b584c14c1c21746f1234457996b62b7c
MATCH?    : True


In [9]:
# Set target round to 10 (the final round)
target.fpga_write(0x0C, [8]) 

# Run AES
target.fpga_write(target.REG_CRYPT_GO, [0x01])
time.sleep(0.01)

# Read both
ct = target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)
snap = target.fpga_read(0x0E, 16)

print(f"Cipherout: {ct.hex().upper()}")
print(f"Snapshot : {snap.hex().upper()}")

Cipherout: B584C14C1C21746F1234457996B62B7C
Snapshot : 49A7577CE539251DAC2AE674BB70F21C


In [10]:
scope.arm()
target.fpga_write(target.REG_CRYPT_TEXTIN, b"\x00"*16)
target.fpga_write(target.REG_CRYPT_GO, b"\x01")

if scope.capture():
    print("❌ Trigger not seen")
else:
    print("✅ Trigger OK")

✅ Trigger OK


In [11]:
scope.glitch.clk_src = "clkgen" 
scope.glitch.output = "clock_xor" 
scope.glitch.repeat = 1
#scope.io.glitch_hp = False 
# scope.gain.gain = 46 
# scope.gain.mode = "high" 
# scope.adc.samples = 500 
# scope.adc.offset = 0 
# scope.adc.basic_mode = "rising_edge" 
# scope.clock.adc_src = "clkgen_x4" 
scope.clock.freq_ctr_src = "clkgen" 
scope.clock.adc_phase = 0 
scope.trigger.triggers = "tio4" 
scope.clock.extclk_freq = 10000000 
scope.clock.clkgen_mul = 5 
scope.clock.clkgen_div = 48 
scope.clock.clkgen_freq = 10000000 
scope.io.hs2 = "glitch" 
scope.glitch.clk_src = 'clkgen' 
scope.glitch.ext_offset = 0 
scope.glitch.trigger_src ="ext_single" #"continuous" #change this depending on glitching desired "ext_single" "continuous" 
scope.glitch.output = "clock_xor" 
scope.glitch.width = 10 
scope.glitch.offset = -20 
#self.api.setParameter(['Glitch Module', 'Output Mode', 'Clock XORd']) 
scope.glitch.repeat = 5 
print("Glitch ready.") 
print("Clock glitch engine armed (but harmless so far)")


Glitch ready.
Clock glitch engine armed (but harmless so far)


In [12]:
scope.io.hs2 = "glitch"
scope.glitch.clk_src = 'clkgen'
scope.glitch.ext_offset = 0
scope.glitch.trigger_src ="continuous" #"continuous" #change this depending on glitching desired   "ext_single"  "continuous"
scope.glitch.output = "clock_xor"
scope.glitch.width = 10
scope.glitch.offset = -20
#self.api.setParameter(['Glitch Module', 'Output Mode', 'Clock XORd'])
scope.glitch.repeat = 5   
print("Glitch ready.")

Glitch ready.


In [13]:
def aes_encrypt_once():
    # fire AES
    target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
    target.fpga_write(target.REG_CRYPT_GO, b"\x01")
    # small wait so ciphertext register updates even if trigger missed
    time.sleep(0.001)
    return bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

def glitch_and_read():
    # arm scope + glitch
    scope.arm()

    # launch AES (this generates the external trigger used by ext_single)
    target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
    target.fpga_write(target.REG_CRYPT_GO, b"\x01")

    # capture waveform (optional but useful for debugging alignment)
    cap_timeout = scope.capture()
    ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

    if cap_timeout:
        return "no_trigger", ct, None

    tr = scope.get_last_trace()

    if ct == EXP_CT:
        return "correct", ct, tr
    else:
        return "faulty", ct, tr

In [16]:
# Pre-compute expected round 8 snapshot (no glitch, clean run)
target.fpga_write(target.REG_CRYPT_KEY, KEY)
target.fpga_write(0x0C, [8])  # set snapshot round to 8
target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
target.fpga_write(target.REG_CRYPT_GO, b"\x01")
time.sleep(0.01)

EXP_SNAP8 = bytes(target.fpga_read(0x0E, 16))
print("Expected Round-8 Snapshot:", EXP_SNAP8.hex().upper())

# Reset snapshot round back to 8 (stays set for all subsequent runs)
target.fpga_write(0x0C, [8])


def glitch_and_read():
    scope.arm()

    target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
    target.fpga_write(target.REG_CRYPT_GO, b"\x01")

    cap_timeout = scope.capture()
    ct    = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))
    snap8 = bytes(target.fpga_read(0x0E, 16))   # <-- read round-8 snapshot

    if cap_timeout:
        return "no_trigger", ct, snap8, None

    tr = np.array(scope.get_last_trace(), dtype=np.float32)

    # Classify what was affected
    ct_ok    = (ct    == EXP_CT)
    snap_ok  = (snap8 == EXP_SNAP8)

    if ct_ok and snap_ok:
        label = "correct"
    elif not snap_ok and not ct_ok:
        label = "faulty_early"   # glitch hit round <=8, corrupted both
    elif snap_ok and not ct_ok:
        label = "faulty_late"    # round 8 was fine, fault hit round 9 or 10
    else:
        label = "faulty_snap_only"  # unusual: snap corrupted but CT somehow correct

    return label, ct, snap8, tr


# --- Main scan loop ---
N_PER_SETTING = 5
records = []
traces  = []
labels  = []
faults  = []
hits    = {"correct": 0, "faulty_early": 0, "faulty_late": 0,
           "faulty_snap_only": 0, "no_trigger": 0}

widths  = range(-49, 49, 2)
offsets = range(-49, 49, 2)

print("Starting Glitch Loop...")

for w in widths:
    scope.glitch.width = w
    for off in offsets:
        scope.glitch.offset = off

        for rep in range(N_PER_SETTING):
            label, ct, snap8, tr = glitch_and_read()

            hits[label] += 1

            if label != "correct" and label != "no_trigger":
                snap_ok = (snap8 == EXP_SNAP8)
                print(
                    f"[{label}] w={w:+d} off={off:+d} | "
                    f"ct={ct.hex().upper()} | "
                    f"snap8={'OK' if snap_ok else snap8.hex().upper()}"
                )
                faults.append({
                    "width": w, "offset": off, "rep": rep,
                    "label": label,
                    "ct_hex": ct.hex(),
                    "snap8_hex": snap8.hex(),
                    "snap8_faulty": not snap_ok,
                })

            records.append({
                "width": w, "offset": off, "rep": rep,
                "label": label,
                "ct_hex": ct.hex(),
                "snap8_hex": snap8.hex(),
            })

            if tr is not None:
                traces.append(tr)
                labels.append(label)

print("\n--- Scan Complete ---")
print("Summary:", hits)
print("Total faults:", len(faults))
print("  Early (<=R8):", sum(1 for f in faults if f["label"] == "faulty_early"))
print("  Late  (R9/10):", sum(1 for f in faults if f["label"] == "faulty_late"))

Expected Round-8 Snapshot: 49A7577CE539251DAC2AE674BB70F21C
Starting Glitch Loop...
[faulty_early] w=-45 off=+1 | ct=A344B045CB3712DD6E71396329C0FDA1 | snap8=CEC43E1A9C9FC9A7C22650601C725535
[faulty_early] w=-39 off=-49 | ct=7A2F4042FEE5F8CFE9BDA9B2ACC0A305 | snap8=BBE255F94E8CB958736B0309CF9395A6
[faulty_late] w=-39 off=+1 | ct=B1846B8C1CB5B76F1B34ECE7DDEE797C | snap8=OK
[faulty_early] w=-37 off=-49 | ct=4D931A1307D259F8FDC80109A80F2E95 | snap8=F545C21DFAAA07750FC3E4A4A31879BA
[faulty_late] w=-33 off=-49 | ct=E971EA9C4CB174281AA1ED5600017977 | snap8=OK
[faulty_early] w=-23 off=+25 | ct=A3D9F078CDEA1A077DC426C1614FF518 | snap8=F5C2E026730C4F98CE2FCA435F384033


(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0a
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0a
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0a
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no t

[faulty_late] w=-15 off=-49 | ct=C0CCEC8A4C1764A4F00B91E9213C7DAF | snap8=OK


(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 08
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 08
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0a
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no t

[faulty_early] w=-5 off=+7 | ct=B2E17F2958840BB52E3CE3759C1BCBDE | snap8=73808A4E44695886A663D82169FBBBAC


(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0a
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0a
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 08
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no t

[faulty_early] w=-3 off=-49 | ct=D7FB8559565D6354079F7BED47C507D1 | snap8=7C9979CF289710B8FAC61DA6B57BB60F


(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0a


KeyboardInterrupt: 

In [ ]:
from Crypto.Cipher import AES as _AES

KEY = bytes.fromhex("2b7e151628aed2a6abf7158809cf4f3c")
PT  = bytes.fromhex("5c692f9103b2302914d7e555e4dcee49")
EXP_CT = _AES.new(KEY, _AES.MODE_ECB).encrypt(PT)

# Tell FPGA to trigger on round 8
target.fpga_write(0x0C, [8])

# Fix your best known width/offset from previous work
scope.glitch.width  = -1
scope.glitch.offset = 47
scope.glitch.repeat = 1       # start with 1 — less aggressive
scope.glitch.trigger_src = "ext_single"

results = []

for ext_off in range(0, 200, 1):   # sweep 0→200 cycles
    scope.glitch.ext_offset = ext_off

    hits = {"correct": 0, "faulty": 0, "no_trigger": 0}

    for rep in range(20):           # 20 trials per offset
        scope.arm()
        target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
        target.fpga_write(target.REG_CRYPT_GO, b"\x00")
        target.fpga_write(target.REG_CRYPT_GO, b"\x01")

        timed_out = scope.capture()
        ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

        if timed_out:
            label = "no_trigger"
        elif ct == EXP_CT:
            label = "correct"
        else:
            label = "faulty"
            delta = bytes(a ^ b for a, b in zip(ct, EXP_CT))
            byteflips = sum(a != b for a, b in zip(ct, EXP_CT))
            print(f"FAULT ext_off={ext_off} byteflips={byteflips} ΔC={delta.hex()}")

        hits[label] += 1

    results.append({"ext_offset": ext_off, **hits})

print("Sweep done")

In [ ]:
import numpy as np
import json

traces_np = np.vstack(traces)  # (num_captured, samples)

np.savez(
    "cw305_glitch_traces_3rd.npz",
    traces=traces_np,
    labels=np.array(labels, dtype="U10"),
    records_json=np.array([json.dumps(r) for r in records], dtype=object),
)

print("Saved: cw305_glitch_traces2nd.npz")
